# Weekly baseline — liquidation trade filter

**Reviewer path:** start with [`REVIEWER_GUIDE.md`](../REVIEWER_GUIDE.md) — inputs → features → LGBM → PnL → conclusions.

**Goal:** Polars load → trade classifier → **Score**, **PnL_kept**, **PnL_filtered**, **turnover/day** (≥500k USD/day).

Data: [Google Drive 6-month archive](https://drive.google.com/file/d/1XmxRsElei-vE8Gc5tkKs2wH4FJVRTevS/view)


## 1. Setup

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import polars as pl
from IPython.display import Markdown, display

ROOT = Path('..').resolve()
if not (ROOT / 'config.py').exists():
    ROOT = Path('.').resolve()
sys.path.insert(0, str(ROOT))

from config import (
    DATA,
    MIN_TURNOVER_PER_DAY,
    RESULTS,
    SYMBOLS,
    TAUS_SEC,
    TRAIN_END,
    TRAIN_START,
    VAL_END,
    has_raw_data,
)
from lib.load_data import data_inventory, load_trades

print('ROOT:', ROOT)
print('DATA:', DATA, '| exists:', DATA.is_dir())
print('Raw parquet OK:', has_raw_data())
print('Turnover constraint: >=', f'${MIN_TURNOVER_PER_DAY:,.0f}/day')


## Reviewer checklist

| Step | Document | Content |
|------|----------|--------|
| 1 | `TASK.md` | Task formulas |
| 2 | `docs/FEATURES.md` | 21 LGBM input features |
| 3 | `docs/LGBM_PARAMS.md` | LightGBM hyperparameters |
| 4 | `docs/BASELINE.md` | What baseline means |
| 5 | `results/lgbm_direction_metrics.json` + ROC PNG | AUC / hit rate |
| 6 | `results/baseline_metrics_full.csv` | Final PnL metrics |


## 3a. LGBM features & parameters

Production filter **`lgbm_nk_30s`**: 21 z-scored features (no `kalman_*`), target = `sign(Δ kalman_mu @ 30s)`.

See `docs/FEATURES.md` and `docs/LGBM_PARAMS.md`.


In [ ]:
import json
from IPython.display import Image, display

metrics = json.loads((ROOT / 'results' / 'lgbm_direction_metrics.json').read_text())
for row in metrics['symbols']:
    print(f"{row['symbol'].upper()} @ 30s: AUC={row['auc_test']:.3f}, hit_rate={row['hit_rate_test']:.3f}, n_test={row['test_rows']:,}")
    print('  features:', len(row['features']), '| params:', {k: row['lgbm_params'][k] for k in ('num_leaves','learning_rate','n_estimators','max_depth')})
    display(Image(filename=str(ROOT / 'results' / 'figures' / row['roc_figure']), width=480))


## 3b. What is baseline?

**Baseline** = no filter: keep all trades, `f_i=0`, **Score = 0** by definition.

LGBM strategy: predict direction → drop trades where signal aligns with taker (`signal × taker_side > 0`). Details: `docs/BASELINE.md`.


## 2. Data inventory (Polars)

If files are missing, run:

```bash
pip install -r requirements.txt
python scripts/download_data.py --out-dir data/
export LIQUIDATION_DATA_ROOT=$(pwd)/data
```

In [ ]:
inv = data_inventory(SYMBOLS)
display(inv)

In [ ]:
if has_raw_data():
    sym = 'btcusdt'
    sample = load_trades(sym, TRAIN_START, TRAIN_END, max_rows=50_000)
    print(f'Sample trades {sym}:', sample.shape)
    display(sample.head(5))
    print('\nDaily notional (sample):')
    daily = (
        sample.with_columns((pl.col('timestamp') // 86_400_000_000).alias('day'))
        .group_by('day')
        .agg(pl.col('notional').sum().alias('notional_usd'), pl.len().alias('n_trades'))
        .sort('day')
    )
    display(daily.head(10))
else:
    print('No local parquet — skip live load; use bundled results below.')

## 3. Metrics (task definition)

| Metric | Meaning |
|--------|--------|
| **PnL_all(τ)** | Weighted mean maker markout, all trades |
| **PnL_kept(τ)** | Mean on kept trades (`f=0`) |
| **PnL_filtered(τ)** | Mean on filtered trades (`f=1`) |
| **Score(τ)** | `PnL_kept − PnL_all` — maximize |
| **Turnover/day** | `Σ w·(1−f) / days`, `w=min(notional, 100k)` |
| **Constraint** | Turnover/day ≥ **500,000 USD** |

Markout: Binance BBO mid at `t+τ`, maker rebate +0.5 bps, Bybit liq +200ms latency in features.

## 4. Trade classifier

Two baselines in `week_baseline/lib/filter.py`:

1. **Heuristic** — filter when |Bybit liquidation flow 30s| is extreme
2. **ML** — logistic regression predicts toxic markout; probability threshold tuned on train for turnover constraint

Full pipeline also includes **LGBM** (`lgbm_nk_30s`) from `tz_assignment/`.

In [ ]:
from lib.filter import classify_trades, FEATURE_COLS
print('Feature columns:', FEATURE_COLS)
import inspect
print('\n--- classify_trades signature ---')
print(inspect.getsource(classify_trades))

## 5. Full baseline results (bundled)

Pre-computed on full test window via `tz_assignment` + `research/pnl_evaluation` (LGBM + Kalman + baseline).

In [ ]:
full_path = RESULTS / 'baseline_metrics_full.csv'
full_df = pd.read_csv(full_path)
cols = [
    'symbol', 'strategy', 'tau_sec', 'score_bps', 'pnl_kept_bps',
    'pnl_filtered_bps', 'pnl_all_bps', 'turnover_kept_usd_day',
    'meets_turnover_constraint', 'winrate_kept', 'kept_frac',
]
display(full_df[cols].sort_values(['symbol', 'tau_sec', 'score_bps'], ascending=[True, True, False]))

In [ ]:
focus = full_df[(full_df['strategy'].isin(['baseline', 'lgbm_nk_30s'])) & (full_df['tau_sec'] == 30)]
fig, ax = plt.subplots(figsize=(8, 4))
x = range(len(focus))
labels = focus['symbol'] + ' / ' + focus['strategy']
ax.bar(x, focus['score_bps'], color=['#4C72B0', '#55A868', '#4C72B0', '#55A868'])
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15, ha='right')
ax.axhline(0, color='k', lw=0.8)
ax.set_ylabel('Score (bps) @ τ=30s')
ax.set_title('Baseline vs LGBM filter (full test)')
plt.tight_layout()
fig.savefig(RESULTS / 'figures' / 'score_tau30_full.png', dpi=120)
plt.show()

## 6. Polars pipeline (live sample)

Run on subsample (~200k trades) — regenerate with `python scripts/run_baseline.py`.

In [ ]:
sample_path = RESULTS / 'baseline_metrics.csv'
if sample_path.is_file():
    sample_df = pd.read_csv(sample_path)
    display(sample_df[['strategy', 'tau_sec', 'score_bps', 'pnl_kept_bps', 'pnl_filtered_bps',
                       'turnover_kept_usd_day', 'meets_turnover_constraint']])
else:
    print('Run: python scripts/run_baseline.py --symbol btcusdt --max-trades 200000')

In [ ]:
if has_raw_data() and sample_path.is_file():
    sub = sample_df[sample_df['tau_sec'] == 30].copy()
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].bar(sub['strategy'], sub['score_bps'])
    axes[0].set_title('Score @ 30s (Polars sample)')
    axes[0].tick_params(axis='x', rotation=20)
    axes[1].bar(sub['strategy'], sub['turnover_kept_usd_day'] / 1e6)
    axes[1].axhline(MIN_TURNOVER_PER_DAY / 1e6, color='r', ls='--', label='500k constraint')
    axes[1].set_ylabel('Turnover kept (M USD/day)')
    axes[1].legend()
    axes[1].tick_params(axis='x', rotation=20)
    plt.tight_layout()
    fig.savefig(RESULTS / 'figures' / 'polars_sample_metrics.png', dpi=120)
    plt.show()

## 7. Turnover constraint check

In [ ]:
check = full_df[full_df['tau_sec']==30][['symbol','strategy','score_bps','turnover_kept_usd_day','meets_turnover_constraint']]
display(check)
ok = full_df['meets_turnover_constraint'].all()
display(Markdown(f"**All strategies meet 500k/day constraint:** {'YES ✓' if ok else 'NO ✗'}"))

## 8. Conclusions

1. **Data pipeline** — Polars loaders for trades, BBO, Binance/Bybit liquidations (+200ms Bybit latency).
2. **Classifiers** — heuristic (Bybit flow) + logistic ML with turnover-calibrated threshold; production LGBM in `tz_assignment`.
3. **Metrics** — Score, PnL_kept, PnL_filtered, turnover/day computed per task spec.
4. **Constraint** — all reported strategies keep ≥500k USD/day clipped turnover.
5. **Best simple ML baseline (full data, τ=30s):** `lgbm_nk_30s` improves Score vs no-filter on BTC/ETH (see table above).

See also: `results/REPORT.md`, `tz_assignment/results/strategies/PNL_REPORT.md`.